In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity


# ==========================================
# 1. Configuración
# ==========================================

ARCHIVO_DATOS = "calificaciones.csv"
ARCHIVO_MODELO = "modelo_recomendador.pkl"

NUMERO_RECOMENDACIONES = 5


# ==========================================
# 2. Crear datos de ejemplo
# ==========================================

if not os.path.exists(ARCHIVO_DATOS):
    datos_ejemplo = pd.DataFrame({
        "usuario": [
            "Ana", "Ana", "Ana", "Ana",
            "Carlos", "Carlos", "Carlos", "Carlos",
            "Laura", "Laura", "Laura", "Laura",
            "Pedro", "Pedro", "Pedro", "Pedro",
            "Sofía", "Sofía", "Sofía", "Sofía",
            "Andrés", "Andrés", "Andrés", "Andrés"
        ],
        "producto": [
            "Laptop", "Celular", "Audífonos", "Libro",
            "Laptop", "Celular", "Audífonos", "Cámara",
            "Laptop", "Celular", "Cámara", "Libro",
            "Laptop", "Audífonos", "Cámara", "Libro",
            "Celular", "Audífonos", "Cámara", "Libro",
            "Laptop", "Celular", "Audífonos", "Cámara"
        ],
        "calificacion": [
            5, 4, 5, 2,
            4, 5, 4, 5,
            5, 4, 5, 3,
            4, 5, 5, 4,
            2, 5, 4, 5,
            5, 3, 5, 4
        ]
    })

    datos_ejemplo.to_csv(
        ARCHIVO_DATOS,
        index=False,
        encoding="utf-8"
    )

    print(f"Se creó el archivo de ejemplo: {ARCHIVO_DATOS}")


# ==========================================
# 3. Cargar los datos
# ==========================================

try:
    datos = pd.read_csv(
        ARCHIVO_DATOS,
        encoding="utf-8"
    )
except FileNotFoundError:
    print(f"No se encontró el archivo '{ARCHIVO_DATOS}'.")
    exit()


columnas_requeridas = [
    "usuario",
    "producto",
    "calificacion"
]

if not all(columna in datos.columns for columna in columnas_requeridas):
    print("El archivo CSV debe contener estas columnas:")
    print("usuario, producto, calificacion")
    exit()


datos = datos.dropna(
    subset=columnas_requeridas
)

datos["calificacion"] = pd.to_numeric(
    datos["calificacion"],
    errors="coerce"
)

datos = datos.dropna(
    subset=["calificacion"]
)


# ==========================================
# 4. Crear matriz usuario-producto
# ==========================================

matriz_calificaciones = datos.pivot_table(
    index="usuario",
    columns="producto",
    values="calificacion",
    aggfunc="mean",
    fill_value=0
)


# ==========================================
# 5. Calcular similitud entre usuarios
# ==========================================

similitudes = cosine_similarity(
    matriz_calificaciones
)

matriz_similitudes = pd.DataFrame(
    similitudes,
    index=matriz_calificaciones.index,
    columns=matriz_calificaciones.index
)


# ==========================================
# 6. Función de recomendación
# ==========================================

def recomendar_productos(
    usuario,
    cantidad=5,
    calificacion_minima=3.5
):
    if usuario not in matriz_calificaciones.index:
        print(f"\nEl usuario '{usuario}' no existe.")
        print("Usuarios disponibles:")
        print(", ".join(matriz_calificaciones.index))
        return pd.DataFrame()

    calificaciones_usuario = matriz_calificaciones.loc[usuario]

    productos_vistos = calificaciones_usuario[
        calificaciones_usuario > 0
    ].index

    usuarios_similares = matriz_similitudes.loc[usuario].drop(
        usuario
    ).sort_values(
        ascending=False
    )

    puntuaciones = {}

    for usuario_similar, similitud in usuarios_similares.items():

        if similitud <= 0:
            continue

        calificaciones_similar = matriz_calificaciones.loc[
            usuario_similar
        ]

        productos_bien_calificados = calificaciones_similar[
            calificaciones_similar >= calificacion_minima
        ]

        for producto, calificacion in productos_bien_calificados.items():

            if producto in productos_vistos:
                continue

            if producto not in puntuaciones:
                puntuaciones[producto] = {
                    "suma_puntuaciones": 0,
                    "suma_similitudes": 0,
                    "usuarios": 0
                }

            puntuaciones[producto]["suma_puntuaciones"] += (
                similitud * calificacion
            )

            puntuaciones[producto]["suma_similitudes"] += similitud
            puntuaciones[producto]["usuarios"] += 1

    if not puntuaciones:
        print("\nNo se encontraron recomendaciones.")
        return pd.DataFrame()

    recomendaciones = []

    for producto, valores in puntuaciones.items():

        if valores["suma_similitudes"] == 0:
            continue

        puntuacion_final = (
            valores["suma_puntuaciones"]
            / valores["suma_similitudes"]
        )

        recomendaciones.append({
            "producto": producto,
            "puntuacion": puntuacion_final,
            "usuarios_similares": valores["usuarios"]
        })

    recomendaciones = pd.DataFrame(
        recomendaciones
    ).sort_values(
        by=["puntuacion", "usuarios_similares"],
        ascending=False
    ).head(cantidad)

    return recomendaciones.reset_index(
        drop=True
    )


# ==========================================
# 7. Mostrar usuarios similares
# ==========================================

def mostrar_usuarios_similares(usuario, cantidad=3):
    if usuario not in matriz_similitudes.index:
        print(f"El usuario '{usuario}' no existe.")
        return

    similares = matriz_similitudes.loc[usuario].drop(
        usuario
    ).sort_values(
        ascending=False
    ).head(cantidad)

    print("\nUSUARIOS MÁS SIMILARES")
    print("======================")

    for nombre, valor in similares.items():
        print(f"{nombre}: {valor:.3f}")


# ==========================================
# 8. Mostrar información del sistema
# ==========================================

print("\nSISTEMA DE RECOMENDACIÓN")
print("========================")
print(f"Usuarios registrados: {len(matriz_calificaciones.index)}")
print(f"Productos registrados: {len(matriz_calificaciones.columns)}")

print("\nUsuarios disponibles:")
print(", ".join(matriz_calificaciones.index))

print("\nProductos disponibles:")
print(", ".join(matriz_calificaciones.columns))


# ==========================================
# 9. Solicitar usuario
# ==========================================

usuario_seleccionado = input(
    "\nEscribe el nombre del usuario: "
).strip()


# ==========================================
# 10. Mostrar similitudes
# ==========================================

mostrar_usuarios_similares(
    usuario_seleccionado
)


# ==========================================
# 11. Generar recomendaciones
# ==========================================

recomendaciones = recomendar_productos(
    usuario_seleccionado,
    cantidad=NUMERO_RECOMENDACIONES
)

if not recomendaciones.empty:
    print("\nRECOMENDACIONES")
    print("================")

    for posicion, fila in recomendaciones.iterrows():
        print(
            f"{posicion + 1}. "
            f"{fila['producto']} - "
            f"Puntuación: {fila['puntuacion']:.2f} - "
            f"Usuarios similares: "
            f"{int(fila['usuarios_similares'])}"
        )

    recomendaciones.to_csv(
        "recomendaciones.csv",
        index=False,
        encoding="utf-8"
    )

    print(
        "\nLas recomendaciones se guardaron en "
        "'recomendaciones.csv'."
    )


# ==========================================
# 12. Guardar el sistema
# ==========================================

modelo_recomendador = {
    "matriz_calificaciones": matriz_calificaciones,
    "matriz_similitudes": matriz_similitudes
}

joblib.dump(
    modelo_recomendador,
    ARCHIVO_MODELO
)

print(f"Modelo guardado como: {ARCHIVO_MODELO}")

Se creó el archivo de ejemplo: calificaciones.csv

SISTEMA DE RECOMENDACIÓN
Usuarios registrados: 6
Productos registrados: 5

Usuarios disponibles:
Ana, Andrés, Carlos, Laura, Pedro, Sofía

Productos disponibles:
Audífonos, Celular, Cámara, Laptop, Libro

Escribe el nombre del usuario: Pipe
El usuario 'Pipe' no existe.

El usuario 'Pipe' no existe.
Usuarios disponibles:
Ana, Andrés, Carlos, Laura, Pedro, Sofía
Modelo guardado como: modelo_recomendador.pkl
